# 01 — Vectores

**Objetivo:** Entender vectores como listas ordenadas de números, sus operaciones básicas y cómo capturan información multidimensional en contextos de analytics.

## Contexto matemático

Un vector en $\mathbb{R}^n$ es un elemento:

$$\mathbf{v} = \begin{pmatrix} v_1 \\ v_2 \\ \vdots \\ v_n \end{pmatrix}$$

Dos visiones complementarias:
- **Algebraica:** lista ordenada de $n$ números reales.
- **Geométrica:** flecha desde el origen hasta el punto $(v_1, v_2, \ldots, v_n)$.

En growth analytics, un vector de canal puede ser $\mathbf{c} = [\text{sessions},\, \text{CVR},\, \text{activaciones}]$.

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)
print('Setup listo')

## 1 — Operaciones básicas

**Suma:** $\mathbf{u} + \mathbf{v} = (u_i + v_i)$  
**Multiplicación escalar:** $c\mathbf{v} = (c v_i)$  
**Sustracción:** $\mathbf{u} - \mathbf{v} = \mathbf{u} + (-1)\mathbf{v}$

In [ ]:
# Vectores de rendimiento de canal (mes A vs mes B)
# [sessions_k, CVR_%, activaciones]
canal_a = np.array([12_000, 3.5, 420])   # Enero
canal_b = np.array([15_500, 4.1, 635])   # Febrero

print('Canal A (enero)  :', canal_a)
print('Canal B (febrero):', canal_b)

delta = canal_b - canal_a
print('\nDelta mes a mes  :', delta)
print('Canal A escalado (×2):', 2 * canal_a)
print('Promedio mensual :', (canal_a + canal_b) / 2)

## 2 — Normas (magnitudes)

La **norma** mide el 'tamaño' de un vector. Las más usadas:

$$\|\mathbf{v}\|_1 = \sum_i |v_i| \quad\text{(Manhattan)}$$

$$\|\mathbf{v}\|_2 = \sqrt{\sum_i v_i^2} \quad\text{(Euclidiana)}$$

$$\|\mathbf{v}\|_p = \left(\sum_i |v_i|^p\right)^{1/p} \quad\text{(Minkowski)}$$

La norma L2 de la diferencia es la **distancia euclidiana** entre dos vectores, útil para comparar períodos de performance.

In [ ]:
# Normalizar primero (las unidades son distintas: miles vs %, vs unidades)
# Usamos z-score simple para hacerlos comparables
ref = np.array([13_750, 3.8, 527.5])   # promedio de referencia
std = np.array([2_500, 0.6, 150])        # escala típica

a_norm = (canal_a - ref) / std
b_norm = (canal_b - ref) / std

print('a normalizado:', a_norm.round(3))
print('b normalizado:', b_norm.round(3))

for p, label in [(1, 'L1 (Manhattan)'), (2, 'L2 (Euclidiana)'), (3, 'L3')]:
    d = np.linalg.norm(b_norm - a_norm, ord=p)
    print(f'  Distancia {label}: {d:.4f}')

## 3 — Producto punto (dot product)

$$\mathbf{u} \cdot \mathbf{v} = \sum_i u_i v_i = \|\mathbf{u}\|_2 \|\mathbf{v}\|_2 \cos\theta$$

Interpretación:
- **Algebraica:** suma ponderada de componentes.
- **Geométrica:** proyección de $\mathbf{u}$ sobre $\mathbf{v}$ (o viceversa) × magnitud.
- Si $\cos\theta = 1$: vectores paralelos (idéntica dirección).  
- Si $\cos\theta = 0$: vectores ortogonales (independientes).  
- Si $\cos\theta = -1$: vectores antiparalelos.

**Similaridad coseno** = $\cos\theta = \dfrac{\mathbf{u}\cdot\mathbf{v}}{\|\mathbf{u}\|\|\mathbf{v}\|}$

In [ ]:
# ¿Qué tan parecidos son dos meses de performance?
dot = np.dot(a_norm, b_norm)
cos_sim = dot / (np.linalg.norm(a_norm) * np.linalg.norm(b_norm))

print(f'Producto punto      : {dot:.4f}')
print(f'Similitud coseno    : {cos_sim:.4f}')
print(f'Ángulo entre meses  : {np.degrees(np.arccos(np.clip(cos_sim, -1, 1))):.1f}°')

# Producto punto como aplicar un vector de pesos a métricas
# (ej: importancia relativa de cada KPI)
pesos_kpi = np.array([0.2, 0.5, 0.3])   # CVR es el KPI más importante
score_a = np.dot(a_norm, pesos_kpi)
score_b = np.dot(b_norm, pesos_kpi)
print(f'\nScore ponderado enero  : {score_a:.3f}')
print(f'Score ponderado febrero: {score_b:.3f}')

## 4 — Vector unitario y normalización

Un **vector unitario** tiene norma 1:

$$\hat{\mathbf{v}} = \dfrac{\mathbf{v}}{\|\mathbf{v}\|_2}$$

Normalizar elimina la magnitud y conserva únicamente la **dirección**. Es el primer paso para calcular similaridad coseno.

In [ ]:
v = np.array([3.0, 4.0])
v_hat = v / np.linalg.norm(v)
print(f'v       = {v}')
print(f'||v||₂  = {np.linalg.norm(v)}')
print(f'v̂       = {v_hat}')
print(f'||v̂||₂  = {np.linalg.norm(v_hat):.6f}  ← siempre 1')

## 5 — Visualización 2D con flechas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Panel izquierdo: suma de vectores
ax = axes[0]
u = np.array([2.0, 1.0])
v = np.array([1.0, 2.5])
w = u + v

def flecha(ax, vec, origen=(0,0), color='#4361ee', label=''):
    ax.annotate('', xy=origen + vec if hasattr(origen, '__add__') else (origen[0]+vec[0], origen[1]+vec[1]),
                xytext=origen,
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    mx, my = (origen[0]+vec[0]/2, origen[1]+vec[1]/2) if isinstance(origen, tuple) else ((origen+vec/2)[0], (origen+vec/2)[1])
    ax.text(mx+0.1, my+0.1, label, color=color, fontsize=10, fontweight='bold')

o = np.array([0.0, 0.0])
flecha(ax, u, o, '#4361ee', 'u')
flecha(ax, v, o, '#f72585', 'v')
flecha(ax, v, u, '#06d6a0', 'v (trasladado)')
flecha(ax, w, o, '#ff9f1c', 'u+v')
ax.set_xlim(-0.5, 4); ax.set_ylim(-0.5, 4)
ax.set_aspect('equal'); ax.set_title('Suma de vectores (regla del paralelogramo)')
ax.axhline(0, color='#ccc', lw=0.8); ax.axvline(0, color='#ccc', lw=0.8)

# Panel derecho: vector unitario
ax2 = axes[1]
theta = np.linspace(0, 2*np.pi, 300)
ax2.plot(np.cos(theta), np.sin(theta), '--', color='#ccc', lw=1, label='Círculo unitario')
for ang, col, lbl in [(30, '#4361ee', 'v₁'), (110, '#f72585', 'v₂'), (200, '#06d6a0', 'v₃')]:
    raw = np.array([np.cos(np.radians(ang))*2.5, np.sin(np.radians(ang))*1.8])
    unit = raw / np.linalg.norm(raw)
    flecha(ax2, raw, o, col, lbl)
    ax2.plot(*unit, 'o', color=col, ms=8)
ax2.set_xlim(-3, 3); ax2.set_ylim(-3, 3)
ax2.set_aspect('equal'); ax2.set_title('Vectores y sus versiones unitarias (en el círculo)')
ax2.axhline(0, color='#ccc', lw=0.8); ax2.axvline(0, color='#ccc', lw=0.8)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 6 — Distancia entre períodos de performance

Modelamos 6 meses de un canal como vectores 3D $[\text{sessions}, \text{CVR}, \text{activaciones}]$ y calculamos la distancia euclidiana al mes de referencia (promedio).

In [ ]:
np.random.seed(7)
meses = ['Ene','Feb','Mar','Abr','May','Jun']
data = np.array([
    [12000, 3.5, 420],
    [15500, 4.1, 635],
    [13200, 3.8, 502],
    [11800, 3.2, 378],
    [16000, 4.4, 704],
    [14500, 4.0, 580],
], dtype=float)

# Normalizar por columna
mu = data.mean(axis=0)
sigma = data.std(axis=0)
data_n = (data - mu) / sigma
ref_n = data_n.mean(axis=0)   # vector de referencia (cero en datos normalizados)

distancias = [np.linalg.norm(data_n[i] - ref_n) for i in range(6)]

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#4361ee' if d == min(distancias) else '#f72585' if d == max(distancias) else '#adb5bd'
          for d in distancias]
ax.bar(meses, distancias, color=colors, edgecolor='white', width=0.6)
ax.set_title('Distancia L2 al vector de referencia (performance promedio)')
ax.set_ylabel('Distancia euclidiana (espacio normalizado)')
for i, d in enumerate(distancias):
    ax.text(i, d + 0.02, f'{d:.2f}', ha='center', fontsize=9)
ax.legend(handles=[mpatches.Patch(color='#4361ee', label='Más cercano al promedio'),
                   mpatches.Patch(color='#f72585', label='Más alejado del promedio')])
plt.tight_layout()
plt.show()

## Resumen

| Concepto | Fórmula | NumPy |
|---|---|---|
| Norma L2 | $\|\mathbf{v}\|_2 = \sqrt{\sum v_i^2}$ | `np.linalg.norm(v)` |
| Norma L1 | $\|\mathbf{v}\|_1 = \sum |v_i|$ | `np.linalg.norm(v, ord=1)` |
| Producto punto | $\mathbf{u}\cdot\mathbf{v} = \sum u_i v_i$ | `np.dot(u, v)` |
| Similitud coseno | $\cos\theta = \frac{\mathbf{u}\cdot\mathbf{v}}{\|\mathbf{u}\|\|\mathbf{v}\|}$ | cálculo manual |
| Vector unitario | $\hat{\mathbf{v}} = \mathbf{v}/\|\mathbf{v}\|$ | `v / np.linalg.norm(v)` |

**Siguiente:** `02_matrices.ipynb`